In [24]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

# username = "aacuser"
# password = "SNHU1234"

# # Connect to database via CRUD Module
# db = AnimalShelter(username, password)

USER = 'aacuser'
PASS = 'mongoDBisFun2025'
HOST = 'localhost' 
PORT = 27017 
DB = 'aac' 
COL = 'animals' 

db = AnimalShelter(USER, PASS, HOST, PORT, DB, COL)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'GraziosoSalvareLogo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# helper dictionary for update dashboard callback
rescue_type_preferences = {
    'Water': {
        "preferred_breeds": ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland'],
        "preferred_sex": "Intact Female",
        "training_age": [26, 156] # [lower, upper]
    },
    'Mountain or Wilderness': { 
        "preferred_breeds": ['German Shepard', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler'],
        "preferred_sex": "Intact Male",
        "training_age": [26, 156] # [lower, upper]
    },
    'Disaster or Individual Tracking': {
        "preferred_breeds": ['Doberman Pinscher', 'German Shepard', 'Golden Retriever', 'Bloodhound', 'Rottweiler'],
        "preferred_sex": "Intact Male",
        "training_age": [20, 300] # [lower, upper]
    }
}

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(
    html.Img(id='app-logo', 
             # module 7 image link. Code_files one didnt work
             src='data:image/png;base64,{}'.format(encoded_image.decode()),
             style={
                 'width':'180px',
                 'height':'auto'
    })),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    # *** FILTERS ***
    html.Div(
        #FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        id='filter-container',
        children=[
            html.Label('Select Rescue Type: '),
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label':'Water', 'value':'Water'},
                    {'label':'Mountain or Wilderness', 'value':'Mountain or Wilderness'},
                    {'label':'Disaster or Individual Tracking', 'value':'Disaster or Individual Tracking'},
                    {'label':'Reset', 'value':'All'}
                ],
                value='All'
            )
        ]
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                        data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here
                        editable=False,
                        filter_action="native",
                        sort_action="native",
                        sort_mode="multi",
                        column_selectable=False,
                        row_selectable="single",
                        row_deletable=False,
                        selected_columns=[],
                        selected_rows=[],
                        page_action="native",
                        page_current= 0,
                        page_size= 10
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={
             'display' : 'flex',
             'justify-content': 'center',
             'gap': '10px'
         },
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ]),
        html.Br(),
        html.Hr(),
        html.H3("Developer: Jerry Vasquez")
])

#############################################
# Interaction Between Components / Controller
#############################################
    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries

    if filter_type == "All":
        query = {}
    # filter by rescue type with their preferred breeds, sex, and training age
    else:
        query = {"animal_type": "Dog", 
                 "breed": {"$in": rescue_type_preferences[filter_type]["preferred_breeds"]},
                 "sex_upon_outcome": rescue_type_preferences[filter_type]["preferred_sex"],
                 "age_upon_outcome_in_weeks": {
                     "$gte": rescue_type_preferences[filter_type]["training_age"][0],
                     "$lte": rescue_type_preferences[filter_type]["training_age"][1]
                 }
                }
    
    data = pd.DataFrame.from_records(db.read(query))
    data.drop(columns=["_id"], inplace=True)
    
    return data.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    dff = pd.DataFrame.from_dict(viewData)
    
    # if reset or showing whole table, show pie chart of number of dogs for each rescue type
    if dff.shape == df.shape:
        
        rescue_type_counts = {}
        
        for rescue_type, rescue_type_dict in rescue_type_preferences.items():
            rescue_type_counts[rescue_type] = dff["breed"].isin(rescue_type_dict['preferred_breeds']).sum() # num row == num dogs
            
        rescue_type_breakdown_df = pd.DataFrame({
            'Rescue Type': list(rescue_type_counts.keys()),
            'Count': list(rescue_type_counts.values())
        })
            
        fig = px.pie(
            rescue_type_breakdown_df,
            names='Rescue Type',
            values='Count',
            title='Dogs by Rescue Type',
            hole=0.2
        )
        fig.update_traces(textinfo='percent+label')
        return [dcc.Graph(figure=fig)]
    
    # if filter is on, show pie charts of the breeds for selected rescue type
    else: 
        fig = px.pie(
            dff,
            names='breed',                 
            title='Distribution of Breeds',
            hole=0.2
            )
    
        fig.update_traces(textinfo='percent+label')
    
        return [dcc.Graph(figure = fig)]
    
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
#FIXME Add in the code for your geolocation chart

    dff = pd.DataFrame(viewData)
    
    # Default map view for when no row is selected
    if index is None or len(index) == 0:
        # show location for first row (default)
        # first row 
        first_row = dff.iloc[0, :] 
        
        lat = first_row['location_lat']
        lon = first_row['location_long']
        
        return [
            # show first row 
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
                dl.TileLayer(),
                dl.Marker(position=[lat, lon], children=[
                    dl.Tooltip(first_row['name']),
                    dl.Popup(
                    [
                        html.H3(f"Animal Name: {first_row['name']}"),
                        html.P(f"Breed: {first_row['breed']}"),
                        html.P(f"Type: {first_row['animal_type']}")
                    ])
                ])
            ])
        ]
                                                                         
    # if row is selected change map location according to it's long and lat
    else:
        # Get selected row
        selected_row = dff.iloc[index[0], :]

        # Use animal location (lat/lon) 
        lat = selected_row["location_lat"]
        lon = selected_row["location_long"]

        # Create map marker for selected animal
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
                dl.TileLayer(id='base-layer-id'),
                # Marker with tool tip and popup
                
                dl.Marker(position=[lat, lon], 
                    children=[
                        dl.Tooltip(selected_row['name']),
                        dl.Popup([
                            html.H3(f"Animal Name: {selected_row['name']}"),
                            html.P(f"Breed: {selected_row['breed']}"),
                            html.P(f"Type: {selected_row['animal_type']}")
                        ])
                    ])
                ])
            ]



# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(port=8051) 

Dash app running on https://societymiami-noisenavy-3000.codio.io/proxy/8051/
